# Explore the order books

The data scientist's first look at the chain's orders: load, inspect, find what is wrong, draft the clean-up. Run it top to bottom with **Run All**. Every question below has its answer in an output.

In [ ]:
import pandas as pd

pd.__version__

## 1. Load and look

In [ ]:
orders = pd.read_csv("data/orders.csv")
orders.head()

In [ ]:
orders.info()

In [ ]:
orders.describe()

Two numbers in `describe()` should worry you: the **min** and the **max** of `qty`. A dish cannot sell fewer than zero portions, and nobody serves thousands of portions of one dish at one lunch.

## 2. Hunt the mistakes

In [ ]:
orders.isna().sum()

In [ ]:
orders.duplicated().sum()

In [ ]:
orders["dish"].nunique()   # the menu has 12 dishes

In [ ]:
orders["dish"].value_counts().tail(8)

In [ ]:
dates = pd.to_datetime(orders["date"], format="%Y-%m-%d", errors="coerce")
orders.loc[dates.isna(), "date"].unique()[:5]

In [ ]:
orders[orders["qty"] < 0].head()

In [ ]:
orders.nlargest(5, "qty")

## 3. Draft the clean-up

In [ ]:
clean = orders.drop_duplicates()
clean["dish"] = clean["dish"].str.strip().str.title()
clean["date"] = pd.to_datetime(clean["date"], format="%Y-%m-%d", errors="coerce")
clean = clean.dropna(subset=["date", "branch", "qty"])
clean = clean[(clean["qty"] >= 0) & (clean["qty"] <= 500)]
len(orders) - len(clean)

In [ ]:
clean["dish"].nunique()

## 4. One row per day, branch and dish

In [ ]:
daily = clean.groupby(["date", "branch", "dish"], as_index=False).agg(qty=("qty", "sum"))
len(daily)

In [ ]:
monthly = daily.set_index("date").groupby("branch")["qty"].resample("ME").sum().unstack("branch")
monthly.tail(3)

5 branches x 12 dishes x 1,369 days = **82,140** rows. If you got that number, the draft works - and it is still a draft: it lives in one notebook, runs only top to bottom, and has no tests. Step 6 of the lab turns it into a command a pipeline can run.